# Cache Data Analysis for Post-Hoc Reasoning Experiments

This notebook analyzes all cached experiment results and generates comprehensive statistics for model performance across datasets.

In [ ]:
import sys
import os
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Import our custom utilities
from utils.data_loader import CacheDataLoader, DatasetProcessor, ResponseParser
from utils.accuracy_calculator import AccuracyCalculator, ExperimentAnalyzer

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

print("Setup complete!")

## 1. Load All Experiment Data

In [ ]:
# Initialize data loader
loader = CacheDataLoader(cache_dir="../cache")
analyzer = ExperimentAnalyzer()

# Get all experiments
experiments = loader.get_all_experiments()
print(f"Found {len(experiments)} experiments")

# Display summary
exp_df = pd.DataFrame(experiments)
if not exp_df.empty:
    print(f"\nModels: {exp_df['model'].nunique()}")
    print(f"Datasets: {exp_df['dataset'].nunique()}")
    print(f"Splits: {exp_df['split'].nunique()}")
    
    print("\nExperiment breakdown:")
    print(exp_df.groupby(['model', 'dataset']).size().to_string())
else:
    print("No experiments found!")

## 2. Process Train/Test Generations and Calculate Accuracies

In [ ]:
# Process each experiment
successful_experiments = 0
failed_experiments = 0

for exp in experiments:
    try:
        print(f"Processing {exp['model']} - {exp['dataset']}...")
        
        # Load experiment data
        train_gen, test_gen = loader.load_train_test_generations(exp['path'])
        dataset, split_info = loader.load_dataset_and_split(exp['path'])
        
        if train_gen is None or test_gen is None or dataset is None:
            print(f"  - Missing data files, skipping")
            failed_experiments += 1
            continue
        
        # Extract ground truth labels
        questions, labels = DatasetProcessor.extract_labels_from_dataset(
            exp['dataset'], dataset, split_info
        )
        
        if not questions or not labels:
            print(f"  - Could not extract labels, skipping")
            failed_experiments += 1
            continue
        
        # Split into train/test based on split_info or assume first half train, second half test
        if split_info and isinstance(split_info, dict) and 'train_indices' in split_info:
            train_indices = split_info['train_indices']
            test_indices = split_info['test_indices']
            
            train_questions = [questions[i] for i in train_indices]
            train_labels = [labels[i] for i in train_indices]
            test_questions = [questions[i] for i in test_indices]
            test_labels = [labels[i] for i in test_indices]
        else:
            # Fallback: assume train_gen and test_gen have same length as their respective splits
            train_size = len(train_gen) if isinstance(train_gen, list) else 0
            test_size = len(test_gen) if isinstance(test_gen, list) else 0
            
            if train_size + test_size > len(labels):
                print(f"  - Size mismatch: generations {train_size + test_size} vs labels {len(labels)}, skipping")
                failed_experiments += 1
                continue
            
            train_questions = questions[:train_size]
            train_labels = labels[:train_size]
            test_questions = questions[train_size:train_size + test_size]
            test_labels = labels[train_size:train_size + test_size]
        
        # Extract predictions from generations
        train_predictions = [
            ResponseParser.extract_predicted_label(gen, exp['dataset']) 
            for gen in train_gen
        ]
        
        test_predictions = [
            ResponseParser.extract_predicted_label(gen, exp['dataset']) 
            for gen in test_gen
        ]
        
        # Add results to analyzer
        if len(train_predictions) == len(train_labels):
            analyzer.add_result(
                exp['model'], exp['dataset'], 'train',
                train_predictions, train_labels, exp['experiment_hash']
            )
        else:
            print(f"  - Train size mismatch: {len(train_predictions)} vs {len(train_labels)}")
        
        if len(test_predictions) == len(test_labels):
            analyzer.add_result(
                exp['model'], exp['dataset'], 'test',
                test_predictions, test_labels, exp['experiment_hash']
            )
        else:
            print(f"  - Test size mismatch: {len(test_predictions)} vs {len(test_labels)}")
        
        successful_experiments += 1
        print(f"  - Success")
        
    except Exception as e:
        print(f"  - Error: {e}")
        failed_experiments += 1
        continue

print(f"\nProcessed: {successful_experiments} successful, {failed_experiments} failed")

## 3. Display Accuracy Summary

In [ ]:
# Get accuracy summary
accuracy_summary = analyzer.get_accuracy_summary()
print(f"Total results: {len(accuracy_summary)}")

if not accuracy_summary.empty:
    # Display summary statistics
    print("\n=== Accuracy Summary ===")
    display(accuracy_summary.round(3))
    
    # Group by model and dataset
    print("\n=== Average Accuracy by Model-Dataset ===")
    model_dataset_avg = accuracy_summary.groupby(['model', 'dataset'])['accuracy'].mean().round(3)
    display(model_dataset_avg)
    
    # Overall statistics
    print("\n=== Overall Statistics ===")
    print(f"Mean accuracy: {accuracy_summary['accuracy'].mean():.3f}")
    print(f"Std accuracy: {accuracy_summary['accuracy'].std():.3f}")
    print(f"Min accuracy: {accuracy_summary['accuracy'].min():.3f}")
    print(f"Max accuracy: {accuracy_summary['accuracy'].max():.3f}")
else:
    print("No accuracy results available!")

## 4. Generate Accuracy Heatmaps

In [ ]:
# Plot train accuracy heatmap
fig = analyzer.plot_accuracy_heatmap(split_type='train', figsize=(14, 8))
if fig:
    plt.show()
else:
    print("No train data available for heatmap")

In [ ]:
# Plot test accuracy heatmap
fig = analyzer.plot_accuracy_heatmap(split_type='test', figsize=(14, 8))
if fig:
    plt.show()
else:
    print("No test data available for heatmap")

## 5. Per-Class Performance Analysis

In [ ]:
# Get per-class metrics
per_class_df = analyzer.get_per_class_summary()

if not per_class_df.empty:
    print("=== Per-Class Performance Summary ===")
    display(per_class_df.round(3))
    
    # Average performance by class across all models/datasets
    print("\n=== Average Performance by Class ===")
    class_avg = per_class_df.groupby('class')[['precision', 'recall', 'f1']].mean().round(3)
    display(class_avg)
else:
    print("No per-class metrics available!")

## 6. Generate Confusion Matrices

In [ ]:
# Generate confusion matrices for each model-dataset combination
test_results = [r for r in analyzer.results if r['split_type'] == 'test']

if test_results:
    # Calculate number of plots needed
    n_results = len(test_results)
    n_cols = 3
    n_rows = (n_results + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
    if n_results == 1:
        axes = [axes]
    elif n_rows == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    
    for i, result in enumerate(test_results):
        if i >= len(axes):
            break
            
        cm = result['confusion_matrix']
        labels = result['labels']
        title = f"{result['model']}\n{result['dataset']}"
        
        # Plot confusion matrix on subplot
        plt.sca(axes[i])
        if cm.size > 0:
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                       xticklabels=labels, yticklabels=labels)
            plt.title(title)
            plt.xlabel('Predicted')
            plt.ylabel('Actual')
        else:
            plt.text(0.5, 0.5, 'No data', ha='center', va='center')
            plt.title(title)
    
    # Hide empty subplots
    for i in range(n_results, len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    plt.show()
else:
    print("No test results available for confusion matrices!")

## 7. Train vs Test Performance Comparison

In [ ]:
# Compare train vs test performance
if not accuracy_summary.empty:
    # Pivot to get train/test comparison
    comparison = accuracy_summary.pivot_table(
        index=['model', 'dataset'], 
        columns='split_type', 
        values='accuracy'
    )
    
    if 'train' in comparison.columns and 'test' in comparison.columns:
        comparison['train_test_gap'] = comparison['train'] - comparison['test']
        comparison_clean = comparison.dropna()
        
        print("=== Train vs Test Performance ===")
        display(comparison_clean.round(3))
        
        if len(comparison_clean) > 0:
            # Plot train vs test scatter
            plt.figure(figsize=(10, 8))
            plt.scatter(comparison_clean['train'], comparison_clean['test'], 
                       s=100, alpha=0.7)
            
            # Add diagonal line (perfect correlation)
            min_val = min(comparison_clean['train'].min(), comparison_clean['test'].min())
            max_val = max(comparison_clean['train'].max(), comparison_clean['test'].max())
            plt.plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.5)
            
            plt.xlabel('Train Accuracy')
            plt.ylabel('Test Accuracy')
            plt.title('Train vs Test Accuracy by Model-Dataset')
            
            # Add labels for points
            for idx, row in comparison_clean.iterrows():
                plt.annotate(f"{idx[0][:10]}\n{idx[1]}", 
                           (row['train'], row['test']), 
                           xytext=(5, 5), textcoords='offset points',
                           fontsize=8, alpha=0.7)
            
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
        
    else:
        print("Missing train or test data for comparison")
else:
    print("No data available for train/test comparison")

## 8. Export Results

In [ ]:
# Export all results to CSV files
output_dir = "../results/accuracy_matrices"
analyzer.export_results_to_csv(output_dir)

print(f"Results exported to {output_dir}")
print("\nGenerated files:")
for file in Path(output_dir).glob("*.csv"):
    print(f"  - {file.name}")

## 9. Model and Dataset Rankings

In [ ]:
if not accuracy_summary.empty:
    # Model rankings (average across all datasets)
    model_rankings = accuracy_summary.groupby('model')['accuracy'].agg(['mean', 'std', 'count']).round(3)
    model_rankings = model_rankings.sort_values('mean', ascending=False)
    
    print("=== Model Rankings (by average accuracy) ===")
    display(model_rankings)
    
    # Dataset difficulty rankings (lower accuracy = harder)
    dataset_rankings = accuracy_summary.groupby('dataset')['accuracy'].agg(['mean', 'std', 'count']).round(3)
    dataset_rankings = dataset_rankings.sort_values('mean', ascending=True)  # Harder datasets first
    
    print("\n=== Dataset Difficulty Rankings (hardest first) ===")
    display(dataset_rankings)
    
    # Plot model rankings
    plt.figure(figsize=(12, 6))
    plt.bar(range(len(model_rankings)), model_rankings['mean'], 
            yerr=model_rankings['std'], capsize=5)
    plt.xticks(range(len(model_rankings)), model_rankings.index, rotation=45, ha='right')
    plt.ylabel('Average Accuracy')
    plt.title('Model Performance Rankings')
    plt.tight_layout()
    plt.show()
    
    # Plot dataset difficulty
    plt.figure(figsize=(10, 6))
    plt.bar(range(len(dataset_rankings)), dataset_rankings['mean'], 
            yerr=dataset_rankings['std'], capsize=5)
    plt.xticks(range(len(dataset_rankings)), dataset_rankings.index, rotation=45, ha='right')
    plt.ylabel('Average Accuracy')
    plt.title('Dataset Difficulty (Lower = Harder)')
    plt.tight_layout()
    plt.show()

## Summary

This notebook has analyzed all cached experiment data and generated:

1. **Accuracy matrices** showing performance across model-dataset combinations
2. **Confusion matrices** for detailed per-class analysis
3. **Train vs test comparisons** to identify overfitting
4. **Model and dataset rankings** to understand relative performance
5. **Exported CSV files** for further analysis

Key findings can be summarized from the visualizations and tables above.